# f0g1 flux sensitivity vs Stark detuning -- multi-Ω₀ analysis

Reloads three lookup-driven Stark sweeps from the vault, refits the f0g1 spectroscopy from raw H5 (no reliance on cached YAML slopes), and plots the flux sensitivity `|df0g1/dΦ|` vs Stark detuning `Δ = f_coupler(I) - drive_freq` for each drive strength `Ω₀/(2π)`.

Vault entries reproduced here (all from `seb/QEC/2026/05/`):
- **2026-05-16  08:51:15** -- Ω₀/(2π) ≈ 1.0 MHz, Δ in [-7.0, -1.0] MHz, 75 points
- **2026-05-16  23:58:17** -- Ω₀/(2π) ≈ 3.0 MHz, Δ in [-15.0, -3.0] MHz, 101 points
- **2026-05-18  11:15:42** -- Ω₀/(2π) ≈ 6.0 MHz, Δ in [-20.0, -5.0] MHz, 101 points

All sweeps share `I_set = -0.05 mA`, `Φ_set ≈ +0.083 Φ₀`, `I_span = 1 μA`, `I_npts = 4`. Flux anchors: `I(Φ=0)=-0.21 mA`, `I(Φ=0.5)=+0.75 mA`.

Workflow:
1. Parse the three daily vault notes to recover the ordered `data_path` lists, `detunings_MHz`, and the per-sweep metadata (`fit_sign`, baseline slope, flux anchors).
2. For every H5 (baseline + per-Δ) refit a Lorentzian to each current row, take `slope = polyfit(Φ [Φ₀], peaks [MHz], 1)`.
3. Reproduce the three published `|df/dΦ|` vs Δ curves, overlay them, and extract the sweet-spot location `Δ_sweet(Ω₀)` via a parabolic fit near the minimum.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import yaml
from scipy.optimize import curve_fit

from slab.datamanagement import SlabFile

# -- Plot style (matches dual_rail_analysis.ipynb) ---------------------------
plt.rcParams.update({
    'figure.figsize': [5, 3.5],
    'font.size': 13,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.6,
    'legend.frameon': False,
    'legend.fontsize': 11,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

COLORS_OMEGA = {
    1.0: '#1f77b4',
    3.0: '#d62728',
    6.0: '#2ca02c',
}

PLOT_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'plots', 'stark_flux_sensitivity')
os.makedirs(PLOT_DIR, exist_ok=True)

def savefig(fig, name, dpi=200):
    for ext in ['png', 'pdf']:
        fig.savefig(os.path.join(PLOT_DIR, f'{name}.{ext}'), dpi=dpi, bbox_inches='tight')

print('Plot dir:', PLOT_DIR)

## Vault entries to analyze

Each `VaultEntry` points to a single `## HH:MM:SS — ...` section inside the daily lab note. The loader extracts the YAML block and resolves the ordered file list.  `VAULT_ROOT` matches the path printed by `station._initialize_vault_root` (see `experiments/station.py`).

In [ ]:
VAULT_ROOT = Path('H:/Shared drives/SLab/Multimode')
USER       = 'seb'
PROJECT    = 'QEC'

@dataclass
class VaultEntry:
    date: str        # 'YYYY-MM-DD'
    time: str        # 'HH:MM:SS'
    label: str       # plot label, e.g. 'Ω₀/(2π) = 1 MHz'
    Omega0_MHz: float
    md_path: Path = field(init=False)

    def __post_init__(self):
        year, month, _ = self.date.split('-')
        self.md_path = (
            VAULT_ROOT / 'Lab' / USER / PROJECT / year / month / f'{self.date}.md'
        )

ENTRIES = [
    VaultEntry('2026-05-16', '08:51:15', label=r'$\Omega_0/(2\pi)=1$ MHz', Omega0_MHz=1.0),
    VaultEntry('2026-05-16', '23:58:17', label=r'$\Omega_0/(2\pi)=3$ MHz', Omega0_MHz=3.0),
    VaultEntry('2026-05-18', '11:15:42', label=r'$\Omega_0/(2\pi)=6$ MHz', Omega0_MHz=6.0),
]

for e in ENTRIES:
    assert e.md_path.exists(), f'Missing vault note: {e.md_path}'
    print(f'  {e.date} {e.time}  ->  {e.md_path}')

## Parse the YAML block out of a daily note

Each measurement is a markdown section starting with `## HH:MM:SS — <title>` followed by a `> [!note]- parameters` Obsidian callout that wraps a YAML code fence.  Every YAML line is prefixed with `> ` (callout quote).  The parser:

1. Walks the file looking for the `## HH:MM:SS — ` heading whose `timestamp:` matches the requested `date+time` (multiple entries may share a heading time string).
2. Captures the inside of the next ``````yaml ... `````` fence.
3. Strips the leading `> ` from each line and runs `yaml.safe_load`.

In [ ]:
_SECTION_RE = re.compile(r'^## (\d{2}:\d{2}:\d{2}) \u2014 ')

def parse_vault_section(md_path: Path, date: str, time: str) -> dict:
    """Return the YAML dict for the `## HH:MM:SS — ...` section whose `timestamp:` field matches `date+T+time`.

    Multiple sections share a heading time string in these logs (run + post-run logs at
    nearly the same minute), so we disambiguate by the exact YAML `timestamp:` line, not
    just the heading.
    """
    target = f"'{date}T{time}'"
    lines = md_path.read_text(encoding='utf-8').splitlines()

    section_starts = [i for i, ln in enumerate(lines) if _SECTION_RE.match(ln)]
    section_starts.append(len(lines))

    for k in range(len(section_starts) - 1):
        i0, i1 = section_starts[k], section_starts[k + 1]
        block = lines[i0:i1]
        yaml_inside = False
        captured = []
        for ln in block:
            s = ln.lstrip()
            if s.startswith('> ```yaml') or s.startswith('> ```'):
                if not yaml_inside:
                    yaml_inside = True
                    continue
                else:
                    break
            if yaml_inside:
                if ln.startswith('> '):
                    captured.append(ln[2:])
                elif ln == '>':
                    captured.append('')
                else:
                    captured.append(ln)
        if not captured:
            continue
        if any(target in c for c in captured):
            yaml_text = '\n'.join(captured)
            return yaml.safe_load(yaml_text)
    raise KeyError(f'No section with timestamp {target} in {md_path}')

# Smoke test: parse all three entries.
_parsed = {}
for e in ENTRIES:
    info = parse_vault_section(e.md_path, e.date, e.time)
    _parsed[(e.date, e.time)] = info
    n_files = len(info['data_path'])
    n_det   = len(info['parameters']['detunings_MHz'])
    print(f'{e.date} {e.time} : {n_files} files, {n_det} detunings  '
          f'(drive_gain={info["parameters"]["drive_gain"]}, '
          f'Ω₀_est={info["parameters"]["Omega0_est_MHz"]:.3f} MHz)')

## Lorentzian peak fit + slope helpers

Identical to `fit_peak_per_row` / `linear_sensitivity` in `measurement_notebooks/QEC/f0g1_stark_flux_sensitivity_lookup.ipynb` so the refit is byte-for-byte comparable with the cached YAML slopes.  Each H5 row sweeps `f0g1_used[i] + detuning_window`, so `xpts` is 2D and the fit uses the per-row axis.

In [ ]:
def _lorentzian(x, A, x0, gamma, offset):
    return offset + A / (1.0 + ((x - x0) / gamma) ** 2)


def fit_peak_per_row(freqs_2d, signal_2d, sign=1):
    """Lorentzian peak per row. `sign=+1` for upward peak, `-1` for dip.

    `freqs_2d` must broadcast to `signal_2d.shape`. Returns shape `(n_rows,)` peak
    frequencies in MHz; NaN on fit failure.
    """
    freqs_2d = np.asarray(freqs_2d, dtype=float)
    signal_2d = np.asarray(signal_2d, dtype=float)
    if freqs_2d.shape != signal_2d.shape:
        freqs_2d = np.broadcast_to(freqs_2d, signal_2d.shape)
    peaks = np.full(signal_2d.shape[0], np.nan)
    for i in range(signal_2d.shape[0]):
        f_row = freqs_2d[i]
        s = sign * signal_2d[i]
        offset_guess = float(np.median(s))
        amp_guess = float(np.max(s) - offset_guess)
        x0_guess = float(f_row[int(np.argmax(s))])
        gamma_guess = max(float(f_row[-1] - f_row[0]) / 50.0,
                          abs(float(f_row[1] - f_row[0])) * 2.0)
        try:
            popt, _ = curve_fit(
                _lorentzian, f_row, s,
                p0=[amp_guess, x0_guess, gamma_guess, offset_guess],
                maxfev=4000,
            )
            peaks[i] = popt[1]
        except Exception:
            peaks[i] = np.nan
    return peaks


def linear_slope(x, y):
    """`y = m x + b` over finite samples. Returns (m, b) or (NaN, NaN)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan, np.nan
    m, b = np.polyfit(x[mask], y[mask], 1)
    return float(m), float(b)


def make_flux_converter(I_phi0_mA, I_phihalf_mA):
    """Return `current_to_flux(I_mA) -> Φ/Φ₀` and its inverse, plus the slope."""
    phi0_per_mA = 0.5 / (I_phihalf_mA - I_phi0_mA)
    def c2f(I_mA): return (np.asarray(I_mA, dtype=float) - I_phi0_mA) * phi0_per_mA
    def f2c(phi):  return I_phi0_mA + np.asarray(phi, dtype=float) / phi0_per_mA
    return c2f, f2c, phi0_per_mA

## Load + refit a single H5 sweep

Each H5 holds a `(n_currents, n_freqs)` `xpts` / `avgi` pair, plus per-row `coupler_current_sweep_mA` and `f0g1_used_MHz`. We refit each row's Lorentzian, regress `peak [MHz]` vs `flux [Φ₀]`, and return the linear slope.  Slope unit `MHz/Φ₀` matches `stark_slopes_MHz_per_phi0` in the vault YAML.

In [ ]:
def load_h5_sweep(path):
    """Read the fields produced by `PulseProbeF0g1StarkAlwaysOn_*_currentsweep.h5` (or the baseline)."""
    with SlabFile(str(path)) as a:
        out = {
            'xpts':         np.array(a['xpts']),
            'avgi':         np.array(a['avgi']),
            'amps':         np.array(a['amps']),
            'currents_mA':  np.array(a['coupler_current_sweep_mA']),
            'f0g1_used':    np.array(a['f0g1_used_MHz']),
        }
        for opt in ('detuning_MHz', 'drive_freq_fixed_MHz', 'I_anchor_mA',
                    'drive_freq_per_point', 'track_drive'):
            if opt in a:
                out[opt] = np.array(a[opt])
    return out


def slope_from_sweep(sweep, *, fit_sign, fit_channel, current_to_flux):
    """Refit the H5 sweep and return slope MHz/Phi0, slope MHz/mA, and the peaks.

    Matches the measurement notebook: fit a Lorentzian to `np.abs(signal_2d)` row-by-row
    with the recorded `fit_sign` (sign=-1 -> dip in abs(signal)).
    """
    signal = np.abs(sweep[fit_channel])
    peaks = fit_peak_per_row(sweep['xpts'], signal, sign=fit_sign)

    currents_mA = sweep['currents_mA']
    fluxes_phi0 = current_to_flux(currents_mA)

    slope_phi0, intercept_phi0 = linear_slope(fluxes_phi0, peaks)
    slope_mA,   intercept_mA   = linear_slope(currents_mA, peaks)
    return {
        'peaks':              peaks,
        'currents_mA':        currents_mA,
        'fluxes_phi0':        fluxes_phi0,
        'slope_MHz_per_phi0': slope_phi0,
        'slope_MHz_per_mA':   slope_mA,
        'intercept_phi0':     intercept_phi0,
        'intercept_mA':       intercept_mA,
    }


## Pipeline: process one vault entry

Takes a `VaultEntry`, parses the YAML, loads all H5 files (baseline + per-detuning), refits everything, and returns a self-contained dict with the full sensitivity curve.

In [ ]:
def process_entry(entry: VaultEntry, *, verbose: bool = True,
                  fit_sign: int | None = None,
                  fit_channel: str | None = None) -> dict:
    """Process one vault entry.

    `fit_sign` / `fit_channel`: if not None, override whatever is cached in the
    YAML (`params['fit_sign']` / `params['fit_channel']`). Use this when the
    measurement-time sign was wrong for a given run (e.g. dispersive lineshape
    at strong Stark drive).
    """
    info = parse_vault_section(entry.md_path, entry.date, entry.time)
    params = info['parameters']

    flux_anchors = params['flux_anchors']
    c2f, f2c, phi0_per_mA = make_flux_converter(
        I_phi0_mA     = float(flux_anchors['I_PHI0_MA']),
        I_phihalf_mA  = float(flux_anchors['I_PHI_HALF_MA']),
    )

    yaml_fit_sign    = int(params.get('fit_sign', 1))
    yaml_fit_channel = str(params.get('fit_channel', 'avgi'))
    fit_sign_used    = int(fit_sign) if fit_sign is not None else yaml_fit_sign
    fit_channel_used = str(fit_channel) if fit_channel is not None else yaml_fit_channel

    if verbose and fit_sign is not None and fit_sign_used != yaml_fit_sign:
        print(f'  [override] fit_sign: YAML={yaml_fit_sign} -> using {fit_sign_used}')
    if verbose and fit_channel is not None and fit_channel_used != yaml_fit_channel:
        print(f'  [override] fit_channel: YAML={yaml_fit_channel!r} -> using {fit_channel_used!r}')

    detunings = np.asarray(params['detunings_MHz'], dtype=float)
    data_paths = info['data_path']
    assert len(data_paths) == 1 + len(detunings), (
        f"data_path length ({len(data_paths)}) != 1 + len(detunings) "
        f"({1 + len(detunings)}) for {entry.date} {entry.time}")

    # --- baseline (Stark off) ---
    if verbose:
        print(f'[{entry.date} {entry.time}] baseline -> {Path(data_paths[0]).name}')
    baseline_sweep = load_h5_sweep(data_paths[0])
    baseline_fit   = slope_from_sweep(
        baseline_sweep, fit_sign=fit_sign_used, fit_channel=fit_channel_used,
        current_to_flux=c2f,
    )

    # --- per-detuning Stark sweeps ---
    slopes_phi0     = np.full(len(detunings), np.nan)
    slopes_mA       = np.full(len(detunings), np.nan)
    intercepts_phi0 = np.full(len(detunings), np.nan)
    drive_freq_fixed = np.full(len(detunings), np.nan)
    n_files = len(data_paths) - 1

    for k, (det, path) in enumerate(zip(detunings, data_paths[1:])):
        try:
            sweep = load_h5_sweep(path)
        except Exception as exc:
            if verbose:
                print(f'  [{k+1}/{n_files}] det={det:+.2f}  LOAD ERROR: {exc}')
            continue
        res = slope_from_sweep(
            sweep, fit_sign=fit_sign_used, fit_channel=fit_channel_used,
            current_to_flux=c2f,
        )
        slopes_phi0[k]     = res['slope_MHz_per_phi0']
        slopes_mA[k]       = res['slope_MHz_per_mA']
        intercepts_phi0[k] = res['intercept_phi0']
        if 'drive_freq_fixed_MHz' in sweep:
            drive_freq_fixed[k] = float(sweep['drive_freq_fixed_MHz'])
        if verbose and (k % 20 == 0 or k == n_files - 1):
            print(f'  [{k+1:3d}/{n_files}] det={det:+6.2f} MHz  '
                  f'slope={res["slope_MHz_per_phi0"]:+8.2f} MHz/Φ₀')

    return {
        'entry':                  entry,
        'params':                 params,
        'fit_sign':               fit_sign_used,
        'fit_sign_yaml':          yaml_fit_sign,
        'fit_channel':            fit_channel_used,
        'fit_channel_yaml':       yaml_fit_channel,
        'phi0_per_mA':            phi0_per_mA,
        'current_to_flux':        c2f,
        'flux_to_current':        f2c,
        'detunings_MHz':          detunings,
        'drive_freq_fixed_MHz':   drive_freq_fixed,
        'slopes_MHz_per_phi0':    slopes_phi0,
        'slopes_MHz_per_mA':      slopes_mA,
        'intercepts_phi0':        intercepts_phi0,
        'baseline':               {
            **baseline_fit,
            'sweep': baseline_sweep,
        },
        # vault-cached slopes (for sanity-check overlay only)
        'yaml_slopes_phi0':       np.asarray(params['stark_slopes_MHz_per_phi0'], dtype=float),
        'yaml_slope_baseline_phi0': float(params['slope_baseline_MHz_per_phi0']),
    }

## Run all three entries

This is the slow step — ~75 + 101 + 101 ≈ 280 H5 files to load and refit. On the analysis box this is roughly 1 minute end to end.

In [ ]:
# Per-entry fit overrides. Key = Omega0_MHz, value = dict with optional
# 'fit_sign' / 'fit_channel'. Anything missing falls back to the YAML cache.
OVERRIDES = {
    # 6.0: {'fit_sign': -1},          # uncomment + tweak as needed
    # 3.0: {'fit_channel': 'amps'},
}

results = []
for entry in ENTRIES:
    ov = OVERRIDES.get(entry.Omega0_MHz, {})
    print('=' * 72)
    print(f'Processing {entry.date} {entry.time}  ({entry.label})')
    if ov:
        print(f'  overrides: {ov}')
    print('=' * 72)
    res = process_entry(entry, verbose=True, **ov)
    results.append(res)
    print(f'  baseline slope (refit)  = {res["baseline"]["slope_MHz_per_phi0"]:+.3f} MHz/Φ₀')
    print(f'  baseline slope (yaml)   = {res["yaml_slope_baseline_phi0"]:+.3f} MHz/Φ₀')
print('\nAll entries processed.')

## Sanity check: refit vs cached YAML slopes

Both curves should overlay almost exactly: the YAML was produced by the same `fit_peak_per_row` running on the same H5 immediately after acquisition.  A clean overlay means the analysis here is self-consistent with what the measurement notebook emitted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, res in zip(axes, results):
    det = res['detunings_MHz']
    ax.plot(det, np.abs(res['yaml_slopes_phi0']), 'k.-', alpha=0.5, label='YAML cache')
    ax.plot(det, np.abs(res['slopes_MHz_per_phi0']), 'C1.', label='refit (H5)')
    ax.set_xlabel(r'Stark detuning $\Delta$ [MHz]')
    ax.set_ylabel(r'$|df_{0g1}/d\Phi|$ [MHz/$\Phi_0$]')
    ax.set_title(res['entry'].label.replace('$', '').replace('\\Omega_0', 'Ω₀').replace('\\pi', 'π'))
    ax.set_ylim(0, 220)
    ax.legend()
plt.tight_layout()
savefig(fig, 'sanity_refit_vs_yaml')
plt.show()

## Reproduce the three published sensitivity plots

Same axes/format as the figures saved under `H:/Shared drives/SLab/Multimode/Lab/seb/QEC/2026/05/figures/`. The dashed horizontal line is the Stark-off baseline `df0g1/dΦ ≈ +133.04 MHz/Φ₀` from Step 2.

In [ ]:
for res in results:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    det = res['detunings_MHz']
    sens = np.abs(res['slopes_MHz_per_phi0'])
    baseline = abs(res['baseline']['slope_MHz_per_phi0'])
    ax.plot(det, sens, 'o-', color=COLORS_OMEGA.get(res['entry'].Omega0_MHz, 'C0'),
            markersize=4, linewidth=1.2,
            label='Step 4 -- drive tracks coupler')
    ax.axhline(baseline, color='k', ls='--',
               label=f'Step 2 baseline {baseline:+.2f} MHz/Φ₀')
    flux_step_phi0 = res['params'].get('flux_step_phi0', float('nan'))
    ax.set_xlabel(r'Stark detuning  $\Delta = f_{\rm coupler}(I) - f_{\rm drive}$  [MHz]')
    ax.set_ylabel(r'$|df_{0g1}/d\Phi|$  [MHz/$\Phi_0$]')
    ax.set_ylim(0, 200)
    ax.set_title(
        f'f0g1 flux sensitivity vs Stark detuning\n'
        f'Ω₀/(2π) ≈ {res["entry"].Omega0_MHz:.2f} MHz   '
        f'Φ_set ≈ {res["params"]["phi_set_phi0"]:+.3f} Φ₀   '
        f'flux step = {flux_step_phi0:.2e} Φ₀'
    )
    ax.legend(loc='upper left')
    plt.tight_layout()
    savefig(fig, f'sensitivity_Omega0_{res["entry"].Omega0_MHz:.1f}MHz')
    plt.show()

## Overlay: all three Ω₀ on one axis

Reading: as `Ω₀` grows the sweet spot (where `|df/dΦ| → 0`) shifts to more negative `Δ` (the drive must be parked further below the coupler to cancel the bare flux sensitivity). The Stark-off baseline `+133 MHz/Φ₀` sets the scale: any point below the dashed line is *less* flux-sensitive than the unperturbed coupler.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))

# Common baseline (same I_set / flux anchors across all three entries).
baseline_phi0 = np.mean([abs(r['baseline']['slope_MHz_per_phi0']) for r in results])
ax.axhline(baseline_phi0, color='k', ls='--', alpha=0.6,
           label=f'Stark-off baseline {baseline_phi0:+.1f} MHz/Φ₀')

for res in results:
    det = res['detunings_MHz']
    sens = np.abs(res['slopes_MHz_per_phi0'])
    color = COLORS_OMEGA.get(res['entry'].Omega0_MHz, None)
    ax.plot(det, sens, 'o-', color=color, markersize=4, linewidth=1.2,
            label=res['entry'].label)

ax.set_xlabel(r'Stark detuning  $\Delta = f_{\rm coupler}(I) - f_{\rm drive}$  [MHz]')
ax.set_ylabel(r'$|df_{0g1}/d\Phi|$  [MHz/$\Phi_0$]')
ax.set_ylim(0, 220)
ax.set_title('f0g1 flux sensitivity vs Stark detuning -- all Ω₀ overlay')
ax.legend(loc='upper right')
plt.tight_layout()
savefig(fig, 'sensitivity_overlay_all_Omega0')
plt.show()

## Sweet-spot extraction

For each Ω₀ we want the detuning at which `|df/dΦ|` is minimized. Strategy:

1. Locate the index of the minimum of `|slope|`.
2. Take a small symmetric window (default ±5 points) around it.
3. Fit a parabola to `(slope_signed)²` vs `Δ` in that window — squaring removes the absolute-value cusp and gives a smooth, well-conditioned minimum.
4. Report the parabola's vertex `Δ_sweet` and the residual sensitivity at that vertex.

Squaring before fitting matters: `|slope|` is V-shaped at the sweet spot so a parabolic fit directly to `|slope|` is biased; `slope²` is locally quadratic in `Δ - Δ_sweet` (since `slope` crosses zero linearly), so the parabolic fit is unbiased near the zero crossing.

In [ ]:
def fit_sweet_spot(det_MHz, slope_phi0, *, window=5):
    """Parabolic fit to `slope²` in a window around the minimum of `|slope|`.

    Returns `(det_sweet, residual_slope_at_sweet, window_idx, parabola_coeffs)` where
    `parabola_coeffs = (a, b, c)` for `a x² + b x + c`. `residual_slope_at_sweet` is
    `sqrt(max(min_value, 0))` so it carries MHz/Φ₀ units.
    """
    det = np.asarray(det_MHz, dtype=float)
    slope = np.asarray(slope_phi0, dtype=float)
    finite = np.isfinite(slope)
    if finite.sum() < 3:
        return np.nan, np.nan, slice(0, 0), (np.nan, np.nan, np.nan)

    k = int(np.nanargmin(np.abs(slope)))
    lo = max(0, k - window)
    hi = min(len(det), k + window + 1)
    sel = slice(lo, hi)
    x = det[sel]
    y = slope[sel] ** 2
    mask = np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan, sel, (np.nan, np.nan, np.nan)
    a, b, c = np.polyfit(x[mask], y[mask], 2)
    if a <= 0:
        return np.nan, np.nan, sel, (a, b, c)
    det_sweet = -b / (2.0 * a)
    min_val   = c - b**2 / (4.0 * a)
    residual  = float(np.sqrt(max(min_val, 0.0)))
    return float(det_sweet), residual, sel, (float(a), float(b), float(c))


sweet = []
for res in results:
    det_sweet, residual, sel, coeffs = fit_sweet_spot(
        res['detunings_MHz'], res['slopes_MHz_per_phi0'], window=5,
    )
    sweet.append({
        'Omega0_MHz':  res['entry'].Omega0_MHz,
        'det_sweet':   det_sweet,
        'residual':    residual,
        'window':      sel,
        'parabola':    coeffs,
        'entry':       res['entry'],
    })
    print(f'{res["entry"].label}:  Δ_sweet = {det_sweet:+.3f} MHz   '
          f'residual |df/dΦ| = {residual:.2f} MHz/Φ₀')

### Visualize the parabolic fit for each sweet spot

Confirms the chosen window straddles a clean zero crossing of the signed slope; a bad fit would show the parabola pulled off-axis by an asymmetric window.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=False)
for ax, res, s in zip(axes, results, sweet):
    det = res['detunings_MHz']
    slope = res['slopes_MHz_per_phi0']
    color = COLORS_OMEGA.get(res['entry'].Omega0_MHz, 'C0')
    ax.plot(det, np.abs(slope), '.', color=color, alpha=0.4, label='all points')
    if isinstance(s['window'], slice) and s['window'].stop > s['window'].start:
        ax.plot(det[s['window']], np.abs(slope[s['window']]), 'o', color=color,
                label='fit window')
    if np.isfinite(s['det_sweet']):
        x_dense = np.linspace(
            det[s['window']].min(), det[s['window']].max(), 200)
        a, b, c = s['parabola']
        ax.plot(x_dense, np.sqrt(np.maximum(a*x_dense**2 + b*x_dense + c, 0.0)),
                '-', color='k', lw=1, label='parabola(slope²)')
        ax.axvline(s['det_sweet'], color='k', ls=':',
                   label=f'Δ_sweet = {s["det_sweet"]:+.2f} MHz')
    ax.set_xlabel(r'Stark detuning $\Delta$ [MHz]')
    ax.set_ylabel(r'$|df_{0g1}/d\Phi|$ [MHz/$\Phi_0$]')
    ax.set_title(f'Ω₀/(2π) = {res["entry"].Omega0_MHz:.1f} MHz')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 60)
    # zoom x around the sweet spot
    if np.isfinite(s['det_sweet']):
        ax.set_xlim(s['det_sweet'] - 2.5, s['det_sweet'] + 2.5)
plt.tight_layout()
savefig(fig, 'sweet_spot_parabolic_fits')
plt.show()

## Δ_sweet vs Ω₀

The Stark shift scales with `Ω₀²/Δ`, so for a drive that exactly cancels the flux sensitivity at a fixed `df_coupler/dΦ`, we expect `Δ_sweet ∝ -Ω₀²`. Plot two reference curves — a linear `Δ_sweet = α Ω₀` (drag-only) and a quadratic `Δ_sweet = β Ω₀²` — and fit both to the three measured points.

In [ ]:
Omega_arr     = np.array([s['Omega0_MHz'] for s in sweet], dtype=float)
det_sweet_arr = np.array([s['det_sweet']  for s in sweet], dtype=float)
residual_arr  = np.array([s['residual']   for s in sweet], dtype=float)

mask = np.isfinite(det_sweet_arr)

# Linear `Δ = a*Ω₀` through the origin (no intercept; sweet spot is at Δ=0 when Ω₀=0).
alpha_lin = float(np.sum(det_sweet_arr[mask] * Omega_arr[mask])
                  / np.sum(Omega_arr[mask] ** 2))
# Quadratic `Δ = b*Ω₀²`
beta_quad = float(np.sum(det_sweet_arr[mask] * Omega_arr[mask] ** 2)
                  / np.sum(Omega_arr[mask] ** 4))

print(f'Linear   fit: Δ_sweet ≈ {alpha_lin:+.3f} · Ω₀')
print(f'Quadratic fit: Δ_sweet ≈ {beta_quad:+.4f} · Ω₀²')

Omega_dense = np.linspace(0, max(Omega_arr.max(), 7), 200)

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(Omega_dense, alpha_lin * Omega_dense, '--', color='gray',
        label=fr'linear: $\Delta = {alpha_lin:+.2f}\,\Omega_0$')
ax.plot(Omega_dense, beta_quad * Omega_dense ** 2, ':', color='k',
        label=fr'quadratic: $\Delta = {beta_quad:+.3f}\,\Omega_0^2$')
for s in sweet:
    color = COLORS_OMEGA.get(s['Omega0_MHz'], 'C0')
    ax.plot(s['Omega0_MHz'], s['det_sweet'], 'o', color=color, ms=10,
            label=s['entry'].label)
ax.set_xlabel(r'$\Omega_0/(2\pi)$  [MHz]')
ax.set_ylabel(r'$\Delta_{\rm sweet}$  [MHz]')
ax.set_title('Sweet-spot detuning vs Stark drive strength')
ax.set_xlim(0, max(Omega_arr.max(), 7))
ax.legend(loc='lower left', fontsize=10)
plt.tight_layout()
savefig(fig, 'sweet_spot_vs_Omega0')
plt.show()

## Summary table

One row per dataset: `Ω₀`, parabolic sweet-spot location, residual flux sensitivity at the sweet spot, and the suppression factor vs the Stark-off baseline `|df/dΦ|_0`.

In [ ]:
baseline_phi0 = np.mean([abs(r['baseline']['slope_MHz_per_phi0']) for r in results])
print(f'Baseline (Stark off) |df0g1/dΦ| = {baseline_phi0:.2f} MHz/Φ₀\n')

hdr = f'{"Ω₀/(2π) [MHz]":>14} | {"Δ_sweet [MHz]":>14} | {"|df/dΦ|_min [MHz/Φ₀]":>22} | {"suppression":>12}'
print(hdr)
print('-' * len(hdr))
for s in sweet:
    suppression = (s['residual'] / baseline_phi0) if baseline_phi0 > 0 else np.nan
    print(f'{s["Omega0_MHz"]:>14.2f} | {s["det_sweet"]:>+14.3f} | '
          f'{s["residual"]:>22.3f} | {suppression:>12.3%}')

## Replot Ramsey traces from a detuning sweep

Reads the per-point quick + long Ramsey h5 files logged in the vault
("Stark always-on Ramsey detuning sweep" entries) and refits each long
trace under a chosen envelope model: `exp`, `gauss`, or `best`
(SSR-based envelope selection from
`fitting.decaysin_analysis.fit_decaysin_with_envelope_selection`).


In [ ]:
import h5py
from fitting.decaysin_analysis import fit_decaysin_with_envelope_selection
import fitting.fitting as fitter

# --- vault entry to replot ---
RAMSEY_DATE = '2026-05-19'
RAMSEY_TIME = '10:30:58'

_ramsey_md = VAULT_ROOT / 'Lab' / USER / PROJECT / RAMSEY_DATE.split('-')[0] / RAMSEY_DATE.split('-')[1] / f'{RAMSEY_DATE}.md'
assert _ramsey_md.exists(), f'Missing vault note: {_ramsey_md}'

_ramsey_info = parse_vault_section(_ramsey_md, RAMSEY_DATE, RAMSEY_TIME)
_ramsey_params = _ramsey_info['parameters']

ramsey_detunings_MHz = np.asarray(_ramsey_params['detunings_MHz'], dtype=float)
ramsey_drive_freqs   = np.asarray(_ramsey_params['drive_freqs_MHz'], dtype=float)
ramsey_long_fnames   = list(_ramsey_params['long_fnames'])
ramsey_quick_fnames  = list(_ramsey_params['quick_fnames'])
ramsey_freqs_used    = np.asarray(_ramsey_params['ramsey_freqs_used'], dtype=float)
ramsey_stark_shifts  = np.asarray(_ramsey_params['stark_shifts_MHz'], dtype=float)

print(f'{RAMSEY_DATE} {RAMSEY_TIME}: {len(ramsey_detunings_MHz)} detunings, '
      f'rabi={_ramsey_params["rabi_rate_MHz"]:.2f} MHz, '
      f'I={_ramsey_params["I_mA"]:+.4f} mA')
print('  long_kwargs :', _ramsey_params.get('long_kwargs'))
print('  quick_kwargs:', _ramsey_params.get('quick_kwargs'))


### Loader + refit helper

`load_ramsey_h5(path)` pulls `xpts`, `avgi`, `avgq`, and the cached fits.
`refit_ramsey(xpts, y, model)` re-runs the envelope selection with one
of three policies:

* `'exp'`   -- force exponential envelope
* `'gauss'` -- force gaussian envelope
* `'best'` -- SSR-margin pick (gauss only wins by `gauss_ssr_margin`)


In [ ]:
def load_ramsey_h5(path):
    path = str(path)
    out = {}
    with h5py.File(path, 'r') as f:
        for k in ('xpts', 'avgi', 'avgq', 'amps', 'phases',
                  'fit_avgi', 'fit_avgi_exp', 'fit_avgi_gauss',
                  'fit_avgq', 'fit_avgq_exp', 'fit_avgq_gauss',
                  'fit_ssr_avgi_exp', 'fit_ssr_avgi_gauss',
                  'fit_ssr_avgq_exp', 'fit_ssr_avgq_gauss',
                  'fit_gauss_ssr_margin'):
            if k in f:
                out[k] = np.array(f[k])
        if 'config' in f.attrs:
            import json as _json
            out['config'] = _json.loads(f.attrs['config'])
    return out


def refit_ramsey(xpts, ydata, *, model='best', gauss_ssr_margin=0.05):
    """Refit a Ramsey trace under `model in {'exp','gauss','best'}`.

    Returns dict with keys p (6,), cov (6,6), envelope ('exp'|'gauss'),
    ssr_exp, ssr_gauss, T2_us, T2_err_us, freq_MHz.
    Parameter order matches fitting.fitting.decaysin/gaussdecaysin:
        (yscale, freq, phase_deg, decay, y0, x0)
    """
    out = fit_decaysin_with_envelope_selection(
        xpts, ydata, use_x0=False, gauss_ssr_margin=gauss_ssr_margin,
    )
    if model == 'exp':
        p, cov = out['p_exp'], out['cov_exp']; env = 'exp'
    elif model == 'gauss':
        p, cov = out['p_gauss'], out['cov_gauss']; env = 'gauss'
    elif model == 'best':
        p, cov = out['p'], out['cov']; env = out['envelope']
    else:
        raise ValueError(f'unknown model: {model!r}')

    T2 = float(p[3])
    T2_err = float(np.sqrt(cov[3, 3])) if np.isfinite(cov[3, 3]) else np.nan
    freq = float(p[1])
    return {
        'p': np.asarray(p, dtype=float),
        'cov': np.asarray(cov, dtype=float),
        'envelope': env,
        'ssr_exp': out['ssr_exp'],
        'ssr_gauss': out['ssr_gauss'],
        'T2_us': T2,
        'T2_err_us': T2_err,
        'freq_MHz': freq,
    }


def eval_ramsey_model(xpts, p, envelope):
    if envelope == 'gauss':
        return fitter.gaussdecaysin(xpts, *p)
    return fitter.decaysin(xpts, *p)


### Trace grid: long Ramsey at each detuning

Toggle `RAMSEY_MODEL` between `'exp'`, `'gauss'`, `'best'` to force the
envelope on every panel.  `RAMSEY_CHANNEL` picks `avgi` (default; what
the live run fits) or `avgq`.


In [ ]:
RAMSEY_MODEL   = 'best'    # 'exp' | 'gauss' | 'best'
RAMSEY_CHANNEL = 'avgi'    # 'avgi' | 'avgq'
RAMSEY_KIND    = 'long'    # 'long' | 'quick'

_fnames = ramsey_long_fnames if RAMSEY_KIND == 'long' else ramsey_quick_fnames

# load + refit every point under the chosen model
ramsey_traces = []
ramsey_fits   = []
for k, (det, drive_f, fn) in enumerate(zip(ramsey_detunings_MHz,
                                            ramsey_drive_freqs,
                                            _fnames)):
    try:
        tr = load_ramsey_h5(fn)
    except Exception as exc:
        print(f'  [{k+1:2d}/{len(_fnames)}] det={det:+6.2f}  LOAD ERROR: {exc}')
        ramsey_traces.append(None); ramsey_fits.append(None); continue
    y = tr[RAMSEY_CHANNEL]
    try:
        fit = refit_ramsey(tr['xpts'], y, model=RAMSEY_MODEL)
    except Exception as exc:
        print(f'  [{k+1:2d}/{len(_fnames)}] det={det:+6.2f}  FIT ERROR: {exc}')
        ramsey_traces.append(tr); ramsey_fits.append(None); continue
    ramsey_traces.append(tr); ramsey_fits.append(fit)

n = len(ramsey_traces)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 2.8 * nrows),
                         sharex=False, sharey=False)
axes = np.atleast_2d(axes).flatten()
for ax, det, drive_f, tr, fit in zip(axes, ramsey_detunings_MHz,
                                      ramsey_drive_freqs,
                                      ramsey_traces, ramsey_fits):
    if tr is None:
        ax.set_visible(False); continue
    x = tr['xpts']; y = tr[RAMSEY_CHANNEL]
    ax.plot(x, y, '.', ms=3, alpha=0.7, color='C0', label='data')
    if fit is not None:
        x_dense = np.linspace(x.min(), x.max(), 400)
        y_dense = eval_ramsey_model(x_dense, fit['p'], fit['envelope'])
        ax.plot(x_dense, y_dense, '-', color='C3', lw=1.1,
                label=f"{fit['envelope']}  T2={fit['T2_us']:.2f} us")
    ax.set_title(f'Δ={det:+.2f} MHz   fd={drive_f:.3f}', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlabel('t [us]')
    ax.set_ylabel(RAMSEY_CHANNEL)
for ax in axes[n:]:
    ax.set_visible(False)
fig.suptitle(
    f'{RAMSEY_DATE} {RAMSEY_TIME} -- {RAMSEY_KIND} Ramsey, model={RAMSEY_MODEL}, ch={RAMSEY_CHANNEL}',
    y=1.0)
plt.tight_layout()
plt.show()


### T2 vs detuning under each model

Refits all points under `exp`, `gauss`, and `best` and overlays the
resulting T2(Δ). Useful for telling whether a single envelope can
explain the whole sweep or whether the model choice flips mid-sweep
under `best`.


In [ ]:
_T2_by_model = {}
_env_by_model = {}
for _model in ('exp', 'gauss', 'best'):
    T2s = np.full(len(ramsey_detunings_MHz), np.nan)
    envs = ['' for _ in ramsey_detunings_MHz]
    for k, (tr, fit_cached) in enumerate(zip(ramsey_traces, ramsey_fits)):
        if tr is None:
            continue
        try:
            fit = refit_ramsey(tr['xpts'], tr[RAMSEY_CHANNEL], model=_model)
        except Exception:
            continue
        T2s[k] = fit['T2_us']
        envs[k] = fit['envelope']
    _T2_by_model[_model] = T2s
    _env_by_model[_model] = envs

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for _model, _color, _marker in (('exp', 'C0', 'o'),
                                ('gauss', 'C1', 's'),
                                ('best', 'C3', '^')):
    ax.plot(ramsey_detunings_MHz, _T2_by_model[_model],
            marker=_marker, color=_color, lw=1.0, ms=5, label=_model)
ax.set_xlabel(r'Stark detuning $\Delta$ [MHz]')
ax.set_ylabel(r'$T_2$ [us]')
ax.set_title(f'{RAMSEY_DATE} {RAMSEY_TIME} -- T2 vs detuning, ch={RAMSEY_CHANNEL}')
ax.legend(title='envelope')
plt.tight_layout()
plt.show()

# print which envelope `best` picked at each point
print('best-envelope picks:')
for det, env in zip(ramsey_detunings_MHz, _env_by_model['best']):
    print(f'  Δ={det:+6.2f} MHz -> {env}')
